In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check the GPU status
!nvidia-smi

Sun Jul 19 22:04:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb"

In [4]:
from datetime import datetime
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from peft import PeftModel

# Configurations

In [5]:
# Run configuration
# SRC_LANG = 'en'
# TGT_LANG = 'vi'
SRC_LANG = 'en'
TGT_LANG = 'en'

# Model configuration
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-1K-LoRA-Merged-v260623145250'
# LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-vi-1K-LoRA-v260622154525'
# LORA_CKPT_DIR = 'checkpoint-60'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Merged-v260711104723'
# LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-vi-5K-LoRA-v260719113226'
# LORA_CKPT_DIR = 'checkpoint-120'
MODEL_ID = 'FacebookAI/xlm-roberta-base'
LORA_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-v260711104723'
LORA_CKPT_DIR = 'checkpoint-320'

# Set up the hub merged model ID
if SRC_LANG in MODEL_ID:
    model_id_prefix, model_id_suffix = MODEL_ID.split(SRC_LANG)
elif TGT_LANG in LORA_ID:
    model_id_prefix, model_id_suffix = LORA_ID.split(TGT_LANG)
else:
    raise ValueError("Model and LoRA IDs do not match the specified source language and target language.")
data_size_str = model_id_suffix.split('LoRA')[0].replace('-', '')
hub_merged_model_id = f"{model_id_prefix}{TGT_LANG}-{data_size_str}-LoRA-Addition-v{datetime.now().strftime("%y%m%d%H%M%S")}"
print(f"Hub merged model ID: {hub_merged_model_id}")

Hub merged model ID: alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438


# Model

In [6]:
# Load the base model
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID, device_map='auto')
print("device:", base_model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


device: cuda:0


In [7]:
# Sanity check
for i in range(11):
    print(f"Base model layer-{i} attention value weight norm:", base_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Base model qa_outputs weight norm:", base_model.qa_outputs.weight.norm().item())
print("Base model qa_outputs bias norm:", base_model.qa_outputs.bias.norm().item())

Base model layer-0 attention value weight norm: 28.364133834838867
Base model layer-1 attention value weight norm: 28.845458984375
Base model layer-2 attention value weight norm: 30.10150146484375
Base model layer-3 attention value weight norm: 36.34113693237305
Base model layer-4 attention value weight norm: 38.29389953613281
Base model layer-5 attention value weight norm: 41.62784957885742
Base model layer-6 attention value weight norm: 38.20062255859375
Base model layer-7 attention value weight norm: 38.956634521484375
Base model layer-8 attention value weight norm: 39.158668518066406
Base model layer-9 attention value weight norm: 34.89980697631836
Base model layer-10 attention value weight norm: 31.030277252197266

Base model qa_outputs weight norm: 0.7699453234672546
Base model qa_outputs bias norm: 0.0


In [8]:
# Load and merge the LoRA adapter into the base model
lora_model = PeftModel.from_pretrained(base_model, subfolder=LORA_CKPT_DIR, model_id=LORA_ID)
merged_model = lora_model.merge_and_unload()

adapter_config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

checkpoint-320/adapter_model.safetensors:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

In [9]:
# Sanity check
for i in range(11):
    print(f"Merged model layer-{i} attention value weight norm:", merged_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Merged model qa_outputs weight norm:", merged_model.qa_outputs.weight.norm().item())
print("Merged model qa_outputs bias norm:", merged_model.qa_outputs.bias.norm().item())

Merged model layer-0 attention value weight norm: 28.367761611938477
Merged model layer-1 attention value weight norm: 28.846525192260742
Merged model layer-2 attention value weight norm: 30.105730056762695
Merged model layer-3 attention value weight norm: 36.34391784667969
Merged model layer-4 attention value weight norm: 38.29513931274414
Merged model layer-5 attention value weight norm: 41.63006591796875
Merged model layer-6 attention value weight norm: 38.20429611206055
Merged model layer-7 attention value weight norm: 38.96026611328125
Merged model layer-8 attention value weight norm: 39.161903381347656
Merged model layer-9 attention value weight norm: 34.90373992919922
Merged model layer-10 attention value weight norm: 31.033811569213867

Merged model qa_outputs weight norm: 0.9504339098930359
Merged model qa_outputs bias norm: 0.010722422040998936


In [10]:
# Upload the merged model to Hugging Face
merged_model.push_to_hub(hub_merged_model_id)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.push_to_hub(hub_merged_model_id)

print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...fz495uk/model.safetensors:   4%|4         | 47.1MB / 1.11GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...z/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

  ...mp_pfa0mgz/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Merged model uploaded to: https://huggingface.co/alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438
